In [ ]:


import os, copy, time, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns
from PIL import Image
from collections import defaultdict
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, accuracy_score
)
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from scipy import stats

try:
    from skimage.metrics import structural_similarity as ssim
    HAS_SSIM = True
except ImportError:
    HAS_SSIM = False

# ══════════════════════════════════════════════════════
# 1. CONFIGURATION
# ══════════════════════════════════════════════════════
CONFIG = {
    'result_dir':   './results',
    'out_dir':      './results/densenet121',
    'weights_dir':  './results/weights',
    'img_size':     224,
    'batch_size':   8,
    'num_classes':  2,
    'class_names':  ['Insensitive', 'Sensitive'],
    'num_workers':  0,
    'epochs':       10,
    'lr':           1e-4,
    'weight_decay': 1e-2,
    'seed':         42,
    'bootstrap_n':  1000,
    'dpi':          300,
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
}

for d in [CONFIG['out_dir'], CONFIG['weights_dir']]:
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# ══════════════════════════════════════════════════════
# 2. STYLE
# ══════════════════════════════════════════════════════
def setup_style():
    plt.rcParams.update({
        'font.family':       'Arial',
        'font.size':         10,
        'axes.facecolor':    'white',
        'figure.facecolor':  'white',
        'axes.edgecolor':    '#333333',
        'axes.linewidth':    1.2,
        'axes.grid':         False,
        'xtick.color':       '#333333',
        'ytick.color':       '#333333',
        'text.color':        '#222222',
        'axes.labelcolor':   '#222222',
        'legend.framealpha': 0.9,
        'legend.edgecolor':  '#cccccc',
        'legend.fontsize':   8,
        'pdf.fonttype':      42,
        'ps.fonttype':       42,
    })

setup_style()

PALETTE = [
    '#2166AC','#D6604D','#1A9641','#762A83','#F4A582',
    '#4DAC26','#92C5DE','#E08214','#A6DBA0','#8073AC',
    '#542788','#E66101','#4393C3','#B2182B','#2D7BB6',
    '#FDAE61','#F46D43','#66BD63','#5AAE61','#5E4FA2',
]

# ══════════════════════════════════════════════════════
# 3. I/O HELPERS
# ══════════════════════════════════════════════════════
def out_path(name): return os.path.join(CONFIG['out_dir'], name)

def save_fig(fig, stem):
    fig.savefig(out_path(stem + '.pdf'), format='pdf', bbox_inches='tight', facecolor='white')
    fig.savefig(out_path(stem + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  [fig] {stem}.pdf")

def save_csv(df, stem):
    path = out_path(stem + '.csv')
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  [csv] {stem}.csv")

# ══════════════════════════════════════════════════════
# 4. DATASET
# ══════════════════════════════════════════════════════
class MRIDataset(Dataset):
    def __init__(self, df, transform=None):
        self.data = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, int(row['label']), str(row['patient_id']), row['path']

def get_transforms(train=True):
    base = [
        transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]
    if train:
        aug = [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ]
        return transforms.Compose([base[0]] + aug + base[1:])
    return transforms.Compose(base)

def load_data():
    train_csv = os.path.join(CONFIG['result_dir'], 'train_dataset.csv')
    test_csv  = os.path.join(CONFIG['result_dir'], 'test_dataset.csv')
    if os.path.exists(train_csv) and os.path.exists(test_csv):
        print("Loading existing train/test split...")
        return pd.read_csv(train_csv), pd.read_csv(test_csv)
    print("Splitting dataset.csv into 80% Train, 20% Test...")
    df = pd.read_csv('dataset.csv')
    train_df, test_df = train_test_split(df, test_size=0.2,
                                         random_state=CONFIG['seed'],
                                         stratify=df['label'])
    train_df.to_csv(train_csv, index=False, encoding='utf-8-sig')
    test_df.to_csv(test_csv,   index=False, encoding='utf-8-sig')
    return train_df, test_df

# ══════════════════════════════════════════════════════
# 5. MODEL (DenseNet-121 with simple linear classifier)
# ══════════════════════════════════════════════════════
def build_densenet121(num_classes):
    model = models.densenet121(weights='IMAGENET1K_V1')
    # Replace classifier with a single linear layer (no dropout, no hidden layers)
    model.classifier = nn.Linear(1024, num_classes)
    return model

# ══════════════════════════════════════════════════════
# 6. GRAD-CAM++
# ══════════════════════════════════════════════════════
class UniversalGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = self.gradients = None
        self._disable_inplace(model)
        self._register()

    def _disable_inplace(self, module):
        for child in module.modules():
            if hasattr(child, 'inplace'): child.inplace = False

    def _register(self):
        self.target_layer.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o))
        self.target_layer.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0]))

    def _to_spatial(self, t):
        if t is None: return None
        if t.ndim == 3:
            B, L, C = t.shape; H = W = int(L**0.5)
            t = t.permute(0, 2, 1).reshape(B, C, H, W)
        return t if t.ndim == 4 else None

    def __call__(self, x, class_idx=None, method='gradcam++'):
        self.model.eval()
        self.activations = self.gradients = None
        x = x.to(CONFIG['device']).requires_grad_(True)
        out = self.model(x)
        if class_idx is None: class_idx = out.argmax(1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(out); one_hot[0, class_idx] = 1.
        out.backward(gradient=one_hot, retain_graph=True)
        acts  = self._to_spatial(self.activations)
        grads = self._to_spatial(self.gradients)
        if acts is None or grads is None: return None, class_idx
        if method == 'gradcam++':
            alpha = grads**2 / (2*grads**2 + (grads**3)*acts.sum((2,3), keepdim=True) + 1e-8)
            weights = (alpha * F.relu(grads)).mean((2, 3), keepdim=True)
        else:
            weights = grads.mean((2, 3), keepdim=True)
        cam = F.relu((weights * acts).sum(1, keepdim=True))[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def overlay(self, cam, raw_rgb, alpha=0.45):
        h, w = raw_rgb.shape[:2]
        cam_up = cv2.resize(cam, (w, h))
        hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam_up), cv2.COLORMAP_JET),
                          cv2.COLOR_BGR2RGB)
        return (raw_rgb * (1 - alpha) + hm * alpha).astype(np.uint8), hm

# ══════════════════════════════════════════════════════
# 7. TRAINING
# ══════════════════════════════════════════════════════
def train_model(model, train_loader, use_mixup=True):
    from sklearn.metrics import roc_auc_score
    device = CONFIG['device']
    model  = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'],
                            weight_decay=CONFIG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.CrossEntropyLoss(label_smoothing=0.25)
    weight_path = os.path.join(CONFIG['weights_dir'], 'densenet121.pth')
    best_auc, history = 0., defaultdict(list)
    if use_mixup:
        print("  Mixup enabled (alpha=0.4)")
    for epoch in range(CONFIG['epochs']):
        model.train()
        run_loss = correct = total = 0
        t_probs, t_lbls = [], []
        for imgs, labels, _, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            if use_mixup:
                lam = np.random.beta(0.4, 0.4)
                idx = torch.randperm(imgs.size(0)).to(device)
                mixed_imgs = lam * imgs + (1 - lam) * imgs[idx]
                out  = model(mixed_imgs)
                loss = lam * criterion(out, labels) + (1 - lam) * criterion(out, labels[idx])
                correct += (out.argmax(1) == labels).sum().item()
            else:
                out  = model(imgs)
                loss = criterion(out, labels)
                correct += (out.argmax(1) == labels).sum().item()
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            total    += labels.size(0)
            with torch.no_grad():
                t_probs += torch.softmax(out, 1)[:, 1].cpu().tolist()
                t_lbls  += labels.cpu().tolist()
        train_acc = correct / total
        train_auc = roc_auc_score(t_lbls, t_probs) if len(set(t_lbls)) > 1 else 0.5
        history['train_loss'].append(run_loss / len(train_loader))
        history['train_acc'].append(train_acc)
        history['train_auc'].append(train_auc)
        scheduler.step()
        if train_auc >= best_auc:
            best_auc = train_auc
            torch.save(model.state_dict(), weight_path)
        print(f"  Ep{epoch+1:02d} "
              f"Loss={history['train_loss'][-1]:.4f} "
              f"TrainAcc={train_acc:.3f} "
              f"TrainAUC={train_auc:.3f}"
              + (' ✓' if train_auc >= best_auc else ''))
    return dict(history), weight_path

# ══════════════════════════════════════════════════════
# 8. EVALUATION
# ══════════════════════════════════════════════════════
def _metrics(y, p, pd_):
    if len(set(y)) < 2:
        return dict(acc=0, sen=0, spe=0, pre=0, f1=0, auc=0.5, ap=0,
                    OR=1, OR_lo=0.1, OR_hi=10)
    cm = confusion_matrix(y, pd_)
    tn, fp, fn, tp = (cm.ravel() if cm.size == 4 else [0, 0, 0, 0])
    acc = (tp+tn) / (tp+tn+fp+fn+1e-8)
    sen = tp / (tp+fn+1e-8);  spe = tn / (tn+fp+1e-8)
    pre = tp / (tp+fp+1e-8);  f1  = 2*pre*sen / (pre+sen+1e-8)
    fpr_, tpr_, _ = roc_curve(y, p);  roc_auc = auc(fpr_, tpr_)
    ap = average_precision_score(y, p)
    tp_, fp_, fn_, tn_ = max(tp, 0.5), max(fp, 0.5), max(fn, 0.5), max(tn, 0.5)
    OR = (tp_*tn_) / (fp_*fn_)
    se_log = np.sqrt(1/tp_ + 1/fp_ + 1/fn_ + 1/tn_)
    OR_lo = np.exp(np.log(OR) - 1.96*se_log)
    OR_hi = np.exp(np.log(OR) + 1.96*se_log)
    return dict(acc=acc, sen=sen, spe=spe, pre=pre, f1=f1, auc=roc_auc, ap=ap,
                OR=OR, OR_lo=OR_lo, OR_hi=OR_hi)

def evaluate_model(model, loader, weight_path):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    y_all, p_all, pd_all, pid_all = [], [], [], []
    with torch.no_grad():
        for imgs, labels, pids, _ in loader:
            out     = model(imgs.to(device))
            p_all  += torch.softmax(out, 1)[:, 1].cpu().tolist()
            pd_all += out.argmax(1).cpu().tolist()
            y_all  += labels.tolist()
            pid_all += list(pids)
    y = np.array(y_all); p = np.array(p_all); pd_ = np.array(pd_all)
    metrics = _metrics(y, p, pd_)
    ci = defaultdict(list)
    for _ in range(CONFIG['bootstrap_n']):
        idx = np.random.choice(len(y), len(y), replace=True)
        if len(set(y[idx])) < 2: continue
        m_b = _metrics(y[idx], p[idx], pd_[idx])
        for k, v in m_b.items(): ci[k].append(v)
    ci = {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in ci.items()}
    fpr_, tpr_, _ = roc_curve(y, p)
    pre_, rec_, _ = precision_recall_curve(y, p)
    return dict(metrics=metrics, ci=ci,
                labels=y, probs=p, preds=pd_, pids=pid_all,
                fpr=fpr_, tpr=tpr_, precision=pre_, recall=rec_)

# ══════════════════════════════════════════════════════
# 9. GRAD-CAM PANEL (single model)
# ══════════════════════════════════════════════════════
def plot_gradcam_panel(model, target_layer, loader, weight_path, split='test', n_samples=8):
    device = CONFIG['device']
    model  = model.to(device)
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    cam_gen = UniversalGradCAM(model, target_layer)
    dataset = loader.dataset
    indices = np.random.permutation(len(dataset))[:n_samples*4]
    samples = []
    for idx in indices:
        if len(samples) >= n_samples: break
        img_t, label, pid, img_path = dataset[idx]
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None: continue
        raw_rgb = cv2.resize(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB),
                             (CONFIG['img_size'], CONFIG['img_size']))
        try:
            cam, pred = cam_gen(img_t.unsqueeze(0), class_idx=1)
        except Exception: continue
        if cam is None: continue
        ov, _ = cam_gen.overlay(cam, raw_rgb)
        with torch.no_grad():
            prob = torch.softmax(model(img_t.unsqueeze(0).to(device)), 1)[0, 1].item()
        samples.append((raw_rgb, cam, ov, label, pred, prob, pid))
    if not samples:
        print("  No valid samples for Grad-CAM panel"); return
    n = len(samples)
    fig = plt.figure(figsize=(16, n*3.5))
    fig.suptitle(f'DenseNet-121 — Grad-CAM++ ({split})', fontsize=13, fontweight='bold', y=1.01)
    gs = gridspec.GridSpec(n, 4, figure=fig, hspace=0.06, wspace=0.04)
    col_titles = ['Original', 'Grad-CAM++ Heatmap', 'Overlay', 'Edge-Enhanced']
    for row_i, (raw, cam, ov, lbl, pred, prob, pid) in enumerate(samples):
        gray   = cv2.cvtColor(raw, cv2.COLOR_RGB2GRAY)
        cam_up = cv2.resize(cam, (raw.shape[1], raw.shape[0]))
        edge   = cv2.normalize((gray * cam_up).astype(np.float32), None, 0, 255,
                               cv2.NORM_MINMAX).astype(np.uint8)
        edge_rgb  = cv2.cvtColor(edge, cv2.COLOR_GRAY2RGB)
        data_cols = [raw, cam, ov, edge_rgb]
        cmaps     = [None, 'YlOrRd', None, None]
        for col_i, (data, cmap) in enumerate(zip(data_cols, cmaps)):
            ax = fig.add_subplot(gs[row_i, col_i])
            if cmap: ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
            else:    ax.imshow(data)
            ax.axis('off')
            if row_i == 0: ax.set_title(col_titles[col_i], fontsize=9, fontweight='bold')
            if col_i == 0:
                cor = '✓' if lbl == pred else '✗'
                col = '#1A9641' if lbl == pred else '#D6604D'
                ax.set_ylabel(f"{cor} {pid[:10]}\nTrue:{CONFIG['class_names'][lbl]}\n"
                              f"Pred:{CONFIG['class_names'][pred]} ({prob:.2f})",
                              color=col, fontsize=7, rotation=0, labelpad=72, va='center')
    plt.tight_layout()
    stem = f"01_gradcam_panel_{split}"
    save_fig(fig, stem)

# ══════════════════════════════════════════════════════
# 10. PLOTTING FUNCTIONS (simplified for single model)
# ══════════════════════════════════════════════════════
def plot_roc_single(train_res, test_res):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, res, title in zip(axes, [train_res, test_res], ['Training Set', 'Test Set']):
        ax.plot([0, 1], [0, 1], '--', color='#aaaaaa', lw=1.2, label='Random')
        ax.plot(res['fpr'], res['tpr'], color=PALETTE[0], lw=2,
                label=f"DenseNet-121  AUC={res['metrics']['auc']:.3f}")
        ax.legend(loc='lower right', fontsize=9)
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC — {title}')
    plt.tight_layout()
    save_fig(fig, "roc_curves")

def plot_calibration_single(train_res, test_res):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, res, title in zip(axes, [train_res, test_res], ['Training', 'Test']):
        y, p = res['labels'], res['probs']
        frac, mpred = calibration_curve(y, p, n_bins=10)
        ax.plot(mpred, frac, 'o-', color=PALETTE[0], lw=2, ms=6, label='DenseNet-121')
        ax.plot([0, 1], [0, 1], '--', color='#aaaaaa')
        ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
        ax.set_title(f'Calibration — {title}')
    plt.tight_layout()
    save_fig(fig, "calibration")

def plot_cm_single(train_res, test_res):
    for res, name in [(train_res, 'Train'), (test_res, 'Test')]:
        cm = confusion_matrix(res['labels'], res['preds'])
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=CONFIG['class_names'],
                    yticklabels=CONFIG['class_names'],
                    ax=ax, cbar=False)
        ax.set_title(f'Confusion Matrix — {name} (AUC={res["metrics"]["auc"]:.3f})')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        save_fig(fig, f"confusion_matrix_{name.lower()}")

# ══════════════════════════════════════════════════════
# 11. MAIN PIPELINE
# ══════════════════════════════════════════════════════
def run():
    print(f"\n{'='*65}")
    print("  DenseNet-121 Radiomics Prediction")
    print(f"  Device: {CONFIG['device']}  |  Output: {CONFIG['out_dir']}")
    print(f"{'='*65}\n")

    # Data
    train_df, test_df = load_data()
    train_ds = MRIDataset(train_df, get_transforms(train=True))
    test_ds  = MRIDataset(test_df,  get_transforms(train=False))
    train_loader_aug  = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                                   shuffle=True,  num_workers=CONFIG['num_workers'])
    test_loader       = DataLoader(test_ds,  batch_size=1,
                                   shuffle=False, num_workers=CONFIG['num_workers'])
    train_eval_loader = DataLoader(MRIDataset(train_df, get_transforms(False)),
                                   batch_size=1, shuffle=False,
                                   num_workers=CONFIG['num_workers'])

    # Build model
    model = build_densenet121(CONFIG['num_classes'])
    weight_path = os.path.join(CONFIG['weights_dir'], 'densenet121.pth')

    # Train or load weights
    if os.path.exists(weight_path):
        print("Weights found, skipping training...")
        history = {}
    else:
        print("Starting training...")
        history, weight_path = train_model(model, train_loader_aug, use_mixup=True)

    # Evaluate
    print("Evaluating on train set...")
    train_res = evaluate_model(model, train_eval_loader, weight_path)
    m = train_res['metrics']
    print(f"  Train AUC={m['auc']:.4f}  F1={m['f1']:.4f}  OR={m['OR']:.3f}")

    print("Evaluating on test set...")
    test_res = evaluate_model(model, test_loader, weight_path)
    m = test_res['metrics']
    print(f"  Test  AUC={m['auc']:.4f}  F1={m['f1']:.4f}  OR={m['OR']:.3f}")

    # Grad-CAM
    print("Generating Grad-CAM panels...")
    target_layer = model.features.denseblock4  # Suitable layer for DenseNet
    plot_gradcam_panel(model, target_layer, train_eval_loader, weight_path, 'train', 6)
    plot_gradcam_panel(model, target_layer, test_loader, weight_path, 'test', 6)

    # Plots
    print("Creating evaluation plots...")
    plot_roc_single(train_res, test_res)
    plot_calibration_single(train_res, test_res)
    plot_cm_single(train_res, test_res)

    # Export prediction scores
    all_records = []
    for res, split in [(train_res, 'Train'), (test_res, 'Test')]:
        for pid, true, pred, prob in zip(res['pids'], res['labels'], res['preds'], res['probs']):
            all_records.append({
                'patient_id': pid,
                'true_label': int(true),
                'true_label_name': CONFIG['class_names'][int(true)],
                'pred_label': int(pred),
                'pred_label_name': CONFIG['class_names'][int(pred)],
                'prob_sensitive': round(float(prob), 6),
                'correct': int(true) == int(pred),
                'split': split
            })
    df = pd.DataFrame(all_records)
    save_csv(df, "prediction_scores")



if __name__ == '__main__':
    run()